# Build the before-and-after image dataset

Run top to bottom. Module cells write the files; the run cell builds `dataset/`. Needs internet access.

In [ ]:
%pip install -q numpy pandas pyyaml opencv-python-headless scikit-image scipy rasterio shapely pyproj pystac-client datasets pillow

### `config.yaml`

In [ ]:
%%writefile config.yaml
paths:
  work: runs
  catalog: runs/catalog.csv
  pairs: runs/pairs.csv
  prepared: runs/prepared
  reports: runs/reports

aoi:
  bbox: [73.03, 18.96, 73.09, 19.02]
  datetime: "2021-01-01/2026-03-31"

sources:
  - name: sentinel2
    type: api
    url: https://earth-search.aws.element84.com/v1
    collections: [sentinel-2-l2a]
    modality: optical
    query: {"eo:cloud_cover": {lt: 15}}
    assets: [red, green, blue, nir]
    mask_asset: scl
    mask_classes: [3, 8, 9, 10]
    gsd: 10
    max_items: 300
# Umbra static catalog is large to traverse; enable only when needed
#   - name: umbra
#     type: static
#     url: https://s3.us-west-2.amazonaws.com/umbra-open-data-catalog/stac/catalog.json
#     modality: sar
#     assets: [GEC]
#     gsd: 0.5
#     max_items: 300
#     max_scan: 20000

pairing:
  min_gap_days: 30
  max_gap_days: 4000
  target_gap_days: 365
  windows:
    before: "2021-01-01/2021-03-31"
    after: "2026-01-01/2026-03-31"
  min_aoi_coverage: 0.8
  max_gsd_ratio: 1.25
  max_off_nadir_diff: 10
  same_platform: false
  same_orbit_state: true
  max_cloud: 15
  max_pairs_per_source: 3
  strict_missing: false
  target_res_m: null
  max_pixels: 36000000
  nodata_value: 0
  min_valid_fraction: 0.7
  reg_max_shift_px: 1.5
  reg_repair_limit_px: 8
  reg_min_response: 0.05
  pct_low: 2
  pct_high: 98
  match_histograms: true
  sar_input: amplitude
  dedupe_cell_deg: 0.05

detect:
  method_by_modality: {optical: classical, sar: classical}
  bands: [0, 1, 2]
  blur_sigma: {optical: 1.5, sar: 3.0}
  threshold: {type: robust_z, value: 3.5}
  morph_kernel: 3
  min_area_px: 30
  learned:
    checkpoint: runs/siamunet.pt
    device: cpu
    tile: 256
    overlap: 32
    threshold: 0.5

model:
  in_channels: 3
  widths: [32, 64, 128, 256]

train:
  root: data/LEVIR-CD
  dirs: {a: A, b: B, label: label}
  splits: {train: train, val: val}
  patch: 256
  batch: 8
  epochs: 40
  lr: 0.001
  device: cpu
  workers: 2
  seed: 0
  pos_weight: 5.0
  dice_weight: 1.0
  out: runs/siamunet.pt

vlm:
  provider: openai_compatible
  base_url: https://generativelanguage.googleapis.com/v1beta/openai/
  api_key_env: GEMINI_API_KEY
  model: gemini-flash-latest
  judge_model: gemini-flash-latest
  max_tokens: 700
  min_interval_s: 6
  retries: 5
  backoff_s: 20
  fatal_status: [400, 401, 403, 404]
  chip_px: 384
  context_pad: 0.6
  min_chip_px: 48
  jpeg_quality: 90
  rgb_idx: [0, 1, 2]
  display_pct: [2, 98]

interpret:
  max_regions: 8
  swap_rate: 1.0
  null_rate: 0.5
  judge: true
  seed: 0
  ring_px: 12
  min_prob: 0.5
  min_contrast: 0.2
  reject_on: [null_hallucination, swap_inconsistent, judge_rejected, weak_pixel_evidence]
  summarize: true
  change_types: [new_construction, demolition, road_change, vegetation_gain, vegetation_loss, water_change, vehicle_or_object_movement, ground_disturbance, other]
  nuisance_types: [none, shadow, seasonal_vegetation, cloud, illumination, registration_error, sensor_artifact]
  inverse_types:
    new_construction: demolition
    demolition: new_construction
    vegetation_gain: vegetation_loss
    vegetation_loss: vegetation_gain
    road_change: road_change
    water_change: water_change
    vehicle_or_object_movement: vehicle_or_object_movement
    ground_disturbance: ground_disturbance
    other: other

prompts:
  system: >
    You are an imagery analyst comparing two co-registered satellite images of the same location.
    Report only changes visible in the images. Do not speculate about purpose or ownership.
    If the difference is explainable by illumination, shadow, season, cloud or sensor effects, say so.
    Answer with one JSON object and nothing else.
  describe: >
    Image 1 is BEFORE ({before_date}), image 2 is AFTER ({after_date}), image 3 is AFTER with the
    candidate region outlined. Modality: {modality}. Pixel size about {gsd} m.
    Return JSON with keys: change_detected (bool), change_type (one of {types}),
    object (short noun phrase), before_state, after_state, description (one sentence),
    nuisance (one of {nuisance}), confidence (0 to 1).
  judge: >
    Image 1 is BEFORE ({before_date}), image 2 is AFTER ({after_date}), image 3 is AFTER with the
    region outlined. A candidate claim: {claim}
    Decide whether the images support the claim. Return JSON with keys: supported (bool), reason (one sentence).
  summary: >
    Summarize these verified and uncertain changes between {before_date} and {after_date} in at most five
    sentences, plainly and without speculation. Findings: {findings}

export:
  results_file: RESULTS.md
  top_changes: 10

### `dataset.yaml`

In [ ]:
%%writefile dataset.yaml
output:
  root: dataset
  jpeg_quality: 95
  max_side_px: 1024
  min_panel_px: 480
  banner_px: 34
  gap_px: 12
  font_scale: 0.8
  font_thickness: 2
  missing_date_label: date not published
  display_pct: [2, 98]
  report_file: dataset/DOWNLOAD_REPORT.md
  index_file: dataset/INDEX.csv

sentinel2:
  enabled: true
  work_dir: runs/sites
  pairs_per_site: 1
  match_histograms: false
  source_url: https://registry.opendata.aws/sentinel-2-l2a-cogs/
  license: Copernicus Sentinel data, free, full and open
  sites:
    - name: navi-mumbai-airport
      note: Greenfield airport construction, Navi Mumbai, India
      bbox: [73.03, 18.96, 73.09, 19.02]
      tries:
        - {before: "2019-01-01/2019-03-31", after: "2026-01-01/2026-03-31"}
        - {before: "2018-12-01/2019-04-30", after: "2025-11-01/2026-04-30"}
    - name: western-sydney-airport
      note: Greenfield airport construction (earthworks began September 2018), Badgerys Creek, Australia
      bbox: [150.67, -33.93, 150.76, -33.85]
      tries:
        - {before: "2019-01-01/2019-03-31", after: "2026-01-01/2026-03-31"}
        - {before: "2019-01-01/2019-06-30", after: "2025-09-01/2026-06-30"}
    - name: gerd-dam
      note: Grand Ethiopian Renaissance Dam and reservoir (filling began July 2020), Ethiopia
      bbox: [35.04, 11.15, 35.15, 11.27]
      tries:
        - {before: "2019-01-01/2019-03-31", after: "2026-01-01/2026-03-31"}
        - {before: "2019-11-01/2020-04-30", after: "2025-11-01/2026-04-30"}

public:
  levir_cd:
    enabled: true
    name: LEVIR-CD (LEVIR-CD+ mirror)
    kind: building change, aerial imagery
    hf_repo: blanchon/LEVIR_CDPlus
    hf_split: test
    count: 12
    before_dir: external/LEVIR-CD/A
    after_dir: external/LEVIR-CD/B
    pattern: "*.png"
    tile_px: null
    seed: 0
    source_url: https://justchenhao.github.io/LEVIR/
    license: check the dataset page; the Hugging Face mirror lists the license as unknown
    date_note: LEVIR-CD does not publish per-pair acquisition dates. The dataset paper gives a range of 2002 to 2018 for the whole set.
    how_to_obtain: Download LEVIR-CD from the dataset page, extract so that external/LEVIR-CD/A and external/LEVIR-CD/B exist.
  whu_cd:
    enabled: true
    name: WHU-CD
    kind: building change, aerial imagery, Christchurch New Zealand
    before_dir: external/WHU-CD/2012/image
    after_dir: external/WHU-CD/2016/image
    pattern: "*.tif"
    count: 8
    tile_px: 1024
    seed: 0
    before_date: "2012"
    after_date: "2016"
    source_url: https://gpcv.whu.edu.cn/data/building_dataset.html
    license: check the dataset page
    date_note: Only the acquisition years are published (2012 and 2016).
    how_to_obtain: Download the two-period image data from the dataset page and extract the 2012 and 2016 image folders to external/WHU-CD/2012/image and external/WHU-CD/2016/image (adjust before_dir and after_dir in dataset.yaml if your folders differ).
  oscd:
    enabled: true
    name: OSCD (Onera Satellite Change Detection)
    kind: urban change, Sentinel-2 multispectral
    root: external/OSCD/images
    dirs: [imgs_1_rect, imgs_2_rect]
    bands: [B04, B03, B02]
    dates_file: dates.txt
    count: 10
    source_url: https://ieee-dataport.org/open-access/oscd-onera-satellite-change-detection
    license: change labels CC BY-NC-SA; check the dataset page
    date_note: Acquisition dates are read from each city's dates.txt.
    how_to_obtain: Download the image archive from IEEE DataPort (free account sign-in) and extract it to external/OSCD/images so that each city folder contains imgs_1_rect, imgs_2_rect and dates.txt.
  xbd:
    enabled: true
    name: xBD (xView2)
    kind: building damage, pre and post disaster, very high resolution
    root: external/xBD/train
    per_disaster: 1
    max_disasters: 10
    source_url: https://xview2.org/
    license: non-commercial (CC BY-NC-SA; the version differs between sources, check xview2.org)
    date_note: Capture dates are read from the label JSON metadata.
    how_to_obtain: Register at xview2.org, download the training images and labels, and extract so that external/xBD/train/images and external/xBD/train/labels exist.
  spacenet7:
    enabled: true
    name: SpaceNet 7 (Planet monthly mosaics)
    kind: urban development, 4 m monthly mosaics
    s3_http: https://spacenet-dataset.s3.amazonaws.com
    train_prefix: spacenet/SN7_buildings/train/
    aoi_count: 6
    min_gap_months: 12
    source_url: https://spacenet.ai/sn7-challenge/
    license: CC BY-SA 4.0
    date_note: Monthly mosaics; the date is the year and month of the mosaic.
    how_to_obtain: Public AWS bucket s3://spacenet-dataset/spacenet/SN7_buildings/ (no account needed). Check your internet access to spacenet-dataset.s3.amazonaws.com.

### `common.py`

In [ ]:
%%writefile common.py
import json
import logging
from pathlib import Path
import yaml


def load_config(path):
    with open(path) as f:
        return yaml.safe_load(f)


def ensure(path):
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


def utm_epsg(lon, lat):
    zone = int((lon + 180) // 6) + 1
    return (32600 if lat >= 0 else 32700) + zone


def get_logger(name):
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")
    return logging.getLogger(name)


def dump_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)


def load_json(path):
    with open(path) as f:
        return json.load(f)

### `discover.py`

In [ ]:
%%writefile discover.py
import json
import pandas as pd
import pystac
from pystac_client import Client
from shapely.geometry import box, shape
from common import ensure, get_logger

log = get_logger("discover")


def normalize(item, src):
    p = item.properties
    when = item.datetime or item.common_metadata.start_datetime
    assets = {k: item.assets[k].href for k in src.get("assets", []) if k in item.assets}
    if src.get("mask_asset") and src["mask_asset"] in item.assets:
        assets["_mask"] = item.assets[src["mask_asset"]].href
    return {
        "id": item.id,
        "source": src["name"],
        "modality": src["modality"],
        "collection": item.collection_id,
        "datetime": when.isoformat() if when else None,
        "platform": p.get("platform"),
        "gsd": p.get("gsd", src.get("gsd")),
        "cloud": p.get("eo:cloud_cover"),
        "off_nadir": p.get("view:off_nadir"),
        "sun_elev": p.get("view:sun_elevation"),
        "orbit_state": p.get("sat:orbit_state"),
        "license": p.get("license"),
        "geometry": json.dumps(item.geometry),
        "assets": json.dumps(assets),
        "needed": len(assets) >= len(src.get("assets", [])),
    }


def search_api(src, cfg, dt):
    client = Client.open(src["url"])
    res = client.search(
        collections=src["collections"],
        bbox=cfg["aoi"]["bbox"],
        datetime=dt,
        query=src.get("query"),
        max_items=src["max_items"],
    )
    return list(res.items())


def search_static(src, cfg, dt):
    cat = pystac.Catalog.from_file(src["url"])
    aoi = box(*cfg["aoi"]["bbox"])
    start, end = (pd.to_datetime(x, utc=True) for x in dt.split("/"))
    out, scanned = [], 0
    for item in cat.get_items(recursive=True):
        scanned += 1
        if scanned > src["max_scan"] or len(out) >= src["max_items"]:
            break
        if item.geometry is None or not shape(item.geometry).intersects(aoi):
            continue
        dt = item.datetime or (pd.to_datetime(item.properties.get("start_datetime"), utc=True).to_pydatetime())
        if dt is None or not (start <= pd.Timestamp(dt).tz_convert("UTC") <= end):
            continue
        out.append(item)
    return out


def search_ranges(cfg):
    win = cfg["pairing"].get("windows")
    return list(win.values()) if win else [cfg["aoi"]["datetime"]]


def run(cfg):
    rows = []
    for src in cfg["sources"]:
        fn = search_api if src["type"] == "api" else search_static
        for dt in search_ranges(cfg):
            try:
                items = fn(src, cfg, dt)
            except Exception as e:
                log.error("source %s %s failed: %s", src["name"], dt, e)
                continue
            log.info("%s %s: %d items", src["name"], dt, len(items))
            rows += [normalize(i, src) for i in items]
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df[df["needed"]].drop(columns="needed").drop_duplicates(["source", "id"])
    ensure(cfg["paths"]["work"])
    df.to_csv(cfg["paths"]["catalog"], index=False)
    log.info("catalog: %d items -> %s", len(df), cfg["paths"]["catalog"])
    return df

### `pairing.py`

In [ ]:
%%writefile pairing.py
import hashlib
import itertools
import json
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from rasterio.vrt import WarpedVRT
from rasterio.warp import transform_bounds
from shapely.geometry import box, shape
from skimage.exposure import match_histograms
from common import dump_json, ensure, get_logger, utm_epsg

log = get_logger("pairing")


def both(a, b):
    return not (a is None or b is None or pd.isna(a) or pd.isna(b))


def build_candidates(cat, cfg):
    pc = cfg["pairing"]
    aoi = box(*cfg["aoi"]["bbox"])
    cat = cat.copy()
    cat["dt"] = pd.to_datetime(cat["datetime"], utc=True, errors="coerce")
    cat = cat.dropna(subset=["dt"]).sort_values("dt")
    rows = []
    for src, g in cat.groupby("source"):
        recs = g.to_dict("records")
        geoms = [shape(json.loads(r["geometry"])).intersection(aoi) for r in recs]
        found = []
        idx = range(len(recs))
        win = pc.get("windows")
        if win:
            wa, wb = (tuple(pd.to_datetime(x, utc=True) for x in win[k].split("/")) for k in ("before", "after"))
            combos = itertools.product([i for i in idx if wa[0] <= recs[i]["dt"] <= wa[1]],
                                       [i for i in idx if wb[0] <= recs[i]["dt"] <= wb[1]])
        else:
            combos = itertools.combinations(idx, 2)
        for i, j in combos:
            a, b = recs[i], recs[j]
            gap = (b["dt"] - a["dt"]).total_seconds() / 86400
            if not pc["min_gap_days"] <= gap <= pc["max_gap_days"]:
                continue
            cov = geoms[i].intersection(geoms[j]).area / aoi.area
            if cov < pc["min_aoi_coverage"]:
                continue
            fails = []
            ga, gb = a["gsd"], b["gsd"]
            if both(ga, gb):
                if max(ga, gb) / min(ga, gb) > pc["max_gsd_ratio"]:
                    fails.append("gsd_ratio")
            elif pc["strict_missing"]:
                fails.append("gsd_missing")
            if both(a["off_nadir"], b["off_nadir"]):
                if abs(a["off_nadir"] - b["off_nadir"]) > pc["max_off_nadir_diff"]:
                    fails.append("off_nadir")
            elif pc["strict_missing"]:
                fails.append("off_nadir_missing")
            if pc["same_platform"] and a["platform"] != b["platform"] and not (pd.isna(a["platform"]) and pd.isna(b["platform"])):
                fails.append("platform")
            if pc["same_orbit_state"] and a["modality"] == "sar" and a["orbit_state"] != b["orbit_state"]:
                fails.append("orbit_state")
            for r in (a, b):
                if r["modality"] == "optical" and not pd.isna(r["cloud"]) and r["cloud"] > pc["max_cloud"]:
                    fails.append("cloud")
                    break
            if not fails:
                cl = sum(0 if pd.isna(r["cloud"]) else r["cloud"] for r in (a, b))
                found.append((cov, -cl if win else -abs(gap - pc["target_gap_days"]), gap, a, b))
        found.sort(key=lambda x: (x[0], x[1]), reverse=True)
        for cov, _, gap, a, b in found[: pc["max_pairs_per_source"]]:
            rows.append({"a": a, "b": b, "coverage": cov, "gap_days": gap})
    return rows


def grid_for(cfg, res):
    w, s, e, n = cfg["aoi"]["bbox"]
    epsg = utm_epsg((w + e) / 2, (s + n) / 2)
    minx, miny, maxx, maxy = transform_bounds("EPSG:4326", f"EPSG:{epsg}", w, s, e, n)
    width = int(np.ceil((maxx - minx) / res))
    height = int(np.ceil((maxy - miny) / res))
    return f"EPSG:{epsg}", from_origin(minx, maxy, res, res), width, height


def read_band(href, crs, transform, width, height, resampling, nodata):
    with rasterio.open(href) as src:
        with WarpedVRT(src, crs=crs, transform=transform, width=width, height=height,
                       resampling=resampling, src_nodata=src.nodata, nodata=nodata) as v:
            return v.read(1).astype("float32")


def load_scene(rec, crs, transform, w, h, pc):
    assets = json.loads(rec["assets"])
    keys = [k for k in assets if k != "_mask"]
    bands = np.stack([read_band(assets[k], crs, transform, w, h, Resampling.bilinear, pc["nodata_value"]) for k in keys])
    valid = (bands != pc["nodata_value"]).all(0)
    if "_mask" in assets:
        classes = rec.get("mask_classes") or []
        m = read_band(assets["_mask"], crs, transform, w, h, Resampling.nearest, pc["nodata_value"])
        valid &= ~np.isin(m, classes)
    return bands, valid, keys


def to_working(bands, valid, modality, pc):
    if modality == "sar":
        x = np.maximum(bands, 1e-6)
        k = 20.0 if pc["sar_input"] == "amplitude" else 10.0
        return k * np.log10(x)
    out = np.empty_like(bands)
    for i, b in enumerate(bands):
        lo, hi = np.percentile(b[valid], [pc["pct_low"], pc["pct_high"]])
        out[i] = np.clip((b - lo) / max(hi - lo, 1e-6), 0, 1)
    return out


def gray(x, valid):
    g = x.mean(0).astype("float32")
    g[~valid] = g[valid].mean() if valid.any() else 0
    return g


def phase_shift(ga, gb):
    win = cv2.createHanningWindow((ga.shape[1], ga.shape[0]), cv2.CV_32F)
    (dx, dy), resp = cv2.phaseCorrelate(ga, gb, win)
    return dx, dy, resp


def shift_image(x, dx, dy):
    m = np.float32([[1, 0, -dx], [0, 1, -dy]])
    return np.stack([cv2.warpAffine(b, m, (b.shape[1], b.shape[0]), flags=cv2.INTER_LINEAR, borderValue=0) for b in x])


def verify_and_prepare(cand, cfg):
    pc = cfg["pairing"]
    a, b = cand["a"], cand["b"]
    src = next(s for s in cfg["sources"] if s["name"] == a["source"])
    a["mask_classes"] = b["mask_classes"] = src.get("mask_classes")
    res = pc["target_res_m"] or max(a["gsd"], b["gsd"])
    crs, transform, w, h = grid_for(cfg, res)
    pid = hashlib.sha1(f"{a['id']}|{b['id']}".encode()).hexdigest()[:10]
    meta = {"pair_id": pid, "source": a["source"], "modality": a["modality"], "before_id": a["id"], "after_id": b["id"],
            "before_date": str(a["dt"]), "after_date": str(b["dt"]), "gap_days": cand["gap_days"],
            "aoi_coverage": cand["coverage"], "crs": crs, "res_m": res, "width": w, "height": h,
            "transform": list(transform)[:6], "license": [a.get("license"), b.get("license")], "steps": []}
    if w * h > pc["max_pixels"]:
        return {**meta, "verdict": "reject", "reason": "grid_too_large"}
    try:
        ba, va, keys = load_scene(a, crs, transform, w, h, pc)
        bb, vb, _ = load_scene(b, crs, transform, w, h, pc)
    except Exception as e:
        return {**meta, "verdict": "reject", "reason": f"read_failed:{e}"}
    valid = va & vb
    vf = float(valid.mean())
    meta.update(bands=keys, valid_fraction=vf)
    if vf < pc["min_valid_fraction"]:
        return {**meta, "verdict": "reject", "reason": "valid_fraction"}
    xa, xb = to_working(ba, va, a["modality"], pc), to_working(bb, vb, b["modality"], pc)
    dx, dy, resp = phase_shift(gray(xa, valid), gray(xb, valid))
    shift = float(np.hypot(dx, dy))
    meta.update(shift_px=shift, reg_response=float(resp))
    verdict = "accept"
    if shift > pc["reg_max_shift_px"]:
        if shift > pc["reg_repair_limit_px"] or resp < pc["reg_min_response"]:
            return {**meta, "verdict": "reject", "reason": "registration"}
        xb = shift_image(xb, dx, dy)
        valid &= shift_image(valid[None].astype("float32"), dx, dy)[0] > 0.5
        dx2, dy2, _ = phase_shift(gray(xa, valid), gray(xb, valid))
        meta["shift_after_px"] = float(np.hypot(dx2, dy2))
        if meta["shift_after_px"] > pc["reg_max_shift_px"]:
            return {**meta, "verdict": "reject", "reason": "registration_unrepaired"}
        verdict = "repair"
        meta["steps"].append("translation_correction")
    if a["modality"] == "optical" and pc["match_histograms"]:
        xb = match_histograms(xb, xa, channel_axis=0).astype("float32")
        meta["steps"].append("histogram_match")
    d = ensure(f"{cfg['paths']['prepared']}/{pid}")
    prof = {"driver": "GTiff", "height": h, "width": w, "crs": crs, "transform": transform, "compress": "deflate"}
    for name, arr in (("before", xa), ("after", xb)):
        with rasterio.open(d / f"{name}.tif", "w", count=arr.shape[0], dtype="float32", **prof) as f:
            f.write(arr.astype("float32"))
    with rasterio.open(d / "valid.tif", "w", count=1, dtype="uint8", **prof) as f:
        f.write(valid.astype("uint8")[None])
    lon = np.mean(cfg["aoi"]["bbox"][0::2])
    lat = np.mean(cfg["aoi"]["bbox"][1::2])
    cell = pc["dedupe_cell_deg"]
    meta["site_id"] = f"{round(lon / cell)}_{round(lat / cell)}"
    meta["verdict"] = verdict
    dump_json(meta, d / "pair.json")
    return meta


def run(cfg):
    cat = pd.read_csv(cfg["paths"]["catalog"])
    out = []
    for cand in build_candidates(cat, cfg):
        m = verify_and_prepare(cand, cfg)
        log.info("%s %s %s", m["pair_id"], m["verdict"], m.get("reason", ""))
        out.append({k: v for k, v in m.items() if k not in ("transform", "steps")})
    if not out:
        raise SystemExit("No candidate pairs. Check runs/catalog.csv: each window in pairing.windows needs scenes covering at least min_aoi_coverage of the area with cloud below max_cloud.")
    df = pd.DataFrame(out)
    df.to_csv(cfg["paths"]["pairs"], index=False)
    return df


def accepted_dirs(cfg, only=None):
    df = pd.read_csv(cfg["paths"]["pairs"])
    df = df[df["verdict"].isin(["accept", "repair"])]
    if only:
        df = df[df["pair_id"].isin(only)]
    return [Path(cfg["paths"]["prepared"]) / p for p in df["pair_id"]]

### `pair_io.py`

In [ ]:
%%writefile pair_io.py
from pathlib import Path
import cv2
import numpy as np
from common import dump_json, ensure


def to_u8(arr, pct):
    if arr.dtype == np.uint8:
        return arr
    a = arr.astype("float32")
    lo, hi = np.percentile(a, pct)
    return (np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype("uint8")


def fit(img, max_side):
    s = max_side / max(img.shape[:2])
    return img if s >= 1 else cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)


def with_banner(img, text, oc):
    b = np.full((oc["banner_px"], img.shape[1], 3), 255, "uint8")
    scale = oc["font_scale"]
    while scale > 0.3 and cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, oc["font_thickness"])[0][0] > img.shape[1] - 16:
        scale -= 0.05
    cv2.putText(b, text, (8, int(oc["banner_px"] * 0.7)), cv2.FONT_HERSHEY_SIMPLEX, scale,
                (0, 0, 0), oc["font_thickness"], cv2.LINE_AA)
    return np.vstack([b, img])


def side_by_side(a, b, label_a, label_b, oc):
    h = max(a.shape[0], b.shape[0])
    up = max(1.0, oc["min_panel_px"] / min(a.shape[1] * h / a.shape[0], b.shape[1] * h / b.shape[0]))
    h = round(h * up)
    rs = lambda x: cv2.resize(x, (round(x.shape[1] * h / x.shape[0]), h), interpolation=cv2.INTER_CUBIC)
    a, b = with_banner(rs(a), label_a, oc), with_banner(rs(b), label_b, oc)
    gap = np.full((a.shape[0], oc["gap_px"], 3), 255, "uint8")
    return np.hstack([a, gap, b])


def write_pair(root, rel, before, after, meta, oc):
    d = ensure(Path(root) / rel)
    a, b = fit(before, oc["max_side_px"]), fit(after, oc["max_side_px"])
    q = [cv2.IMWRITE_JPEG_QUALITY, oc["jpeg_quality"]]
    na = oc["missing_date_label"]
    sbs = side_by_side(a, b, f"BEFORE  {meta.get('before_date') or na}", f"AFTER  {meta.get('after_date') or na}", oc)
    for name, img in (("before.jpg", a), ("after.jpg", b), ("side_by_side.jpg", sbs)):
        cv2.imwrite(str(d / name), img[:, :, ::-1], q)
    dump_json(meta, d / "meta.json")
    return d

### `public_sets.py`

In [ ]:
%%writefile public_sets.py
import random
import re
import urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np
import rasterio
from rasterio.windows import Window
from common import load_json
from pair_io import to_u8


def read_rgb(path, pct, tile=None, rng=None, origin=None):
    with rasterio.open(str(path)) as s:
        win = None
        if tile and (s.height > tile or s.width > tile):
            if origin is None:
                origin = (rng.randint(0, max(s.height - tile, 0)), rng.randint(0, max(s.width - tile, 0)))
            win = Window(origin[1], origin[0], min(tile, s.width - origin[1]), min(tile, s.height - origin[0]))
        arr = s.read(window=win)
    arr = arr[:3] if arr.shape[0] >= 3 else np.repeat(arr[:1], 3, axis=0)
    return to_u8(arr.transpose(1, 2, 0), pct), origin


def pick_even(items, count):
    step = max(len(items) // count, 1)
    return items[::step][:count]


def iso8(s):
    return f"{s[:4]}-{s[4:6]}-{s[6:8]}"


def levir_hf(c, pct):
    from datasets import load_dataset
    ds = load_dataset(c["hf_repo"], split=c["hf_split"], streaming=True)
    for i, row in enumerate(ds.take(c["count"])):
        yield {"rel": f"{c['hf_split']}_{i:04d}", "before": np.array(row["image1"].convert("RGB")),
               "after": np.array(row["image2"].convert("RGB")), "meta": {}}


def paired_folders(c, pct):
    bd, ad = Path(c["before_dir"]), Path(c["after_dir"])
    if not bd.is_dir() or not ad.is_dir():
        raise FileNotFoundError(f"folder not found: {bd if not bd.is_dir() else ad}")
    names = sorted(p.name for p in bd.glob(c["pattern"]) if (ad / p.name).exists())
    if not names:
        raise FileNotFoundError(f"no file name matching '{c['pattern']}' exists in both {bd} and {ad}")
    rng = random.Random(c["seed"])
    for name in pick_even(names, c["count"]):
        a, origin = read_rgb(bd / name, pct, c.get("tile_px"), rng)
        b, _ = read_rgb(ad / name, pct, c.get("tile_px"), None, origin)
        yield {"rel": Path(name).stem, "before": a, "after": b, "meta": {"source_file": name}}


def xbd_local(c, pct):
    root = Path(c["root"])
    img_dir, lab_dir = root / "images", root / "labels"
    if not img_dir.is_dir():
        raise FileNotFoundError(f"folder not found: {img_dir}")
    pres = sorted(img_dir.glob("*_pre_disaster.png"))
    if not pres:
        raise FileNotFoundError(f"no *_pre_disaster.png files in {img_dir}")
    groups = {}
    for p in pres:
        groups.setdefault(p.name.split("_")[0], []).append(p)
    for disaster in sorted(groups)[: c["max_disasters"]]:
        for pre in pick_even(groups[disaster], c["per_disaster"]):
            post = pre.with_name(pre.name.replace("_pre_disaster", "_post_disaster"))
            if not post.exists():
                continue
            dates = []
            for p in (pre, post):
                lj = lab_dir / (p.stem + ".json")
                cd = load_json(lj).get("metadata", {}).get("capture_date") if lj.exists() else None
                dates.append(str(cd)[:10] if cd else None)
            a, _ = read_rgb(pre, pct)
            b, _ = read_rgb(post, pct)
            yield {"rel": f"{disaster}/{pre.name.replace('_pre_disaster.png', '')}", "before": a, "after": b,
                   "meta": {"before_date": dates[0], "after_date": dates[1], "disaster": disaster}}


def oscd_local(c, pct):
    root = Path(c["root"])
    if not root.is_dir():
        raise FileNotFoundError(f"folder not found: {root}")
    d1, d2 = c["dirs"]
    cities = sorted(d for d in root.iterdir() if (d / d1).is_dir() and (d / d2).is_dir())
    if not cities:
        raise FileNotFoundError(f"no city folder containing {d1} and {d2} under {root}")
    for city in pick_even(cities, c["count"]):
        imgs = []
        for sub in (d1, d2):
            bands = [rasterio.open(str(city / sub / f"{b}.tif")).read(1) for b in c["bands"]]
            imgs.append(to_u8(np.stack(bands, axis=-1), pct))
        dt = (city / c["dates_file"]).read_text() if (city / c["dates_file"]).exists() else ""
        found = re.findall(r"\d{8}", dt)
        yield {"rel": city.name, "before": imgs[0], "after": imgs[1],
               "meta": {"before_date": iso8(found[0]) if len(found) > 1 else None,
                        "after_date": iso8(found[1]) if len(found) > 1 else None, "city": city.name}}


def s3_list(base, prefix, delimiter=None):
    url = f"{base}/?list-type=2&prefix={prefix}" + (f"&delimiter={delimiter}" if delimiter else "")
    with urllib.request.urlopen(url, timeout=60) as r:
        root = ET.fromstring(r.read())
    ns = {"s": root.tag.split("}")[0].strip("{")}
    return ([e.text for e in root.findall("s:Contents/s:Key", ns)],
            [e.text for e in root.findall("s:CommonPrefixes/s:Prefix", ns)])


def spacenet7_s3(c, pct):
    _, prefixes = s3_list(c["s3_http"], c["train_prefix"], "/")
    aois = sorted(prefixes)[: c["aoi_count"]]
    if not aois:
        raise RuntimeError(f"no AOI folder listed under {c['train_prefix']}")
    for aoi in aois:
        name = aoi.rstrip("/").split("/")[-1]
        keys, _ = s3_list(c["s3_http"], f"{aoi}images/")
        dated = sorted((re.search(r"global_monthly_(\d{4})_(\d{2})", k).groups(), k) for k in keys
                       if k.endswith(".tif") and re.search(r"global_monthly_(\d{4})_(\d{2})", k))
        if len(dated) < 2:
            continue
        months = lambda ym: int(ym[0]) * 12 + int(ym[1])
        first = dated[0]
        later = [d for d in dated if months(d[0]) - months(first[0]) >= c["min_gap_months"]]
        if not later:
            continue
        last = later[-1]
        a, _ = read_rgb(f"{c['s3_http']}/{first[1]}", pct)
        b, _ = read_rgb(f"{c['s3_http']}/{last[1]}", pct)
        yield {"rel": name, "before": a, "after": b,
               "meta": {"before_date": "-".join(first[0]), "after_date": "-".join(last[0]), "aoi": name}}

### `interpret.py`

In [ ]:
%%writefile interpret.py
import base64
import json
import os
import random
import time
from pathlib import Path
import cv2
import numpy as np
import rasterio
from common import dump_json, get_logger, load_json

log = get_logger("interpret")


def parse_json(text):
    i, j = text.find("{"), text.rfind("}")
    if i < 0 or j < 0:
        return None
    try:
        return json.loads(text[i:j + 1])
    except json.JSONDecodeError:
        return None


def to_openai(system, content):
    parts = []
    for c in content:
        if c["type"] == "text":
            parts.append({"type": "text", "text": c["text"]})
        else:
            src = c["source"]
            parts.append({"type": "image_url", "image_url": {"url": f"data:{src['media_type']};base64,{src['data']}"}})
    return [{"role": "system", "content": system}, {"role": "user", "content": parts}]


class Backend:
    def __init__(self, vc):
        self.vc = vc
        self.last = 0.0
        if vc["provider"] == "anthropic":
            import anthropic
            self.client = anthropic.Anthropic(api_key=os.environ[vc["api_key_env"]])
        else:
            from openai import OpenAI
            self.client = OpenAI(base_url=vc["base_url"], api_key=os.environ[vc["api_key_env"]])

    def _call(self, model, system, content, max_tokens):
        if self.vc["provider"] == "anthropic":
            r = self.client.messages.create(model=model, max_tokens=max_tokens, system=system,
                                            messages=[{"role": "user", "content": content}])
            return "".join(b.text for b in r.content if b.type == "text")
        r = self.client.chat.completions.create(model=model, max_tokens=max_tokens, messages=to_openai(system, content))
        return r.choices[0].message.content or ""

    def complete(self, model, system, content, max_tokens):
        for k in range(self.vc["retries"]):
            wait = self.vc["min_interval_s"] - (time.time() - self.last)
            if wait > 0:
                time.sleep(wait)
            try:
                text = self._call(model, system, content, max_tokens)
                self.last = time.time()
                return text
            except Exception as e:
                self.last = time.time()
                if getattr(e, "status_code", None) in self.vc["fatal_status"]:
                    raise
                log.warning("VLM call failed (%s); retry %d of %d", e, k + 1, self.vc["retries"])
                time.sleep(self.vc["backoff_s"] * (k + 1))
        raise RuntimeError("VLM call failed after all retries")


def call_vlm(client, model, system, content, max_tokens):
    return client.complete(model, system, content, max_tokens)


def to_display(arr, modality, vc):
    if modality == "sar":
        g = arr[0]
        lo, hi = np.percentile(g, vc["display_pct"])
        g = np.clip((g - lo) / max(hi - lo, 1e-6), 0, 1)
        rgb = np.stack([g, g, g])
    else:
        rgb = arr[vc["rgb_idx"]]
    return (np.clip(rgb, 0, 1) * 255).astype("uint8").transpose(1, 2, 0)


def window(bbox, shape, vc):
    x0, y0, x1, y1 = bbox
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    half = max(x1 - x0, y1 - y0, vc["min_chip_px"]) * (1 + vc["context_pad"]) / 2
    h, w = shape
    return (max(int(cx - half), 0), max(int(cy - half), 0), min(int(cx + half), w), min(int(cy + half), h))


def chip(img, win, vc, poly=None):
    x0, y0, x1, y1 = win
    c = img[y0:y1, x0:x1].copy()
    scale = vc["chip_px"] / max(c.shape[:2])
    if poly is not None:
        pts = (np.array(poly) - [x0, y0]).astype("int32").reshape(-1, 1, 2)
        cv2.polylines(c, [pts], True, (255, 0, 0), 2)
    c = cv2.resize(c, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    ok, buf = cv2.imencode(".jpg", c[:, :, ::-1], [cv2.IMWRITE_JPEG_QUALITY, vc["jpeg_quality"]])
    return base64.standard_b64encode(buf.tobytes()).decode()


def img_block(b64):
    return {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": b64}}


def build_content(prompt, imgs):
    out = []
    for label, b64 in imgs:
        out += [{"type": "text", "text": label}, img_block(b64)]
    return out + [{"type": "text", "text": prompt}]


def contrast(prob, lab, region, ring):
    x0, y0, x1, y1 = region["bbox_px"]
    h, w = lab.shape
    wx0, wy0, wx1, wy1 = max(x0 - ring - 1, 0), max(y0 - ring - 1, 0), min(x1 + ring + 1, w), min(y1 + ring + 1, h)
    L, P = lab[wy0:wy1, wx0:wx1], prob[wy0:wy1, wx0:wx1]
    inside = L == region["id"]
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * ring + 1,) * 2)
    ringm = (cv2.dilate(inside.astype("uint8"), k) > 0) & (L == 0)
    if not ringm.any():
        return float(P[inside].mean())
    return float(P[inside].mean() - P[ringm].mean())


def evaluate(desc, swap, null, judge, evidence, ic):
    f = []
    if desc is None:
        return ["vlm_parse_failed"], "uncertain"
    if not desc.get("change_detected"):
        f.append("vlm_no_change")
    if swap is not None:
        exp = ic["inverse_types"].get(desc.get("change_type"), desc.get("change_type"))
        if not swap.get("change_detected") or swap.get("change_type") != exp:
            f.append("swap_inconsistent")
    if null is not None and null.get("change_detected"):
        f.append("null_hallucination")
    if evidence["mean_prob"] < ic["min_prob"] or evidence["contrast"] < ic["min_contrast"]:
        f.append("weak_pixel_evidence")
    if desc.get("nuisance") not in (None, "none"):
        f.append("nuisance:" + str(desc.get("nuisance")))
    if judge is not None and not judge.get("supported"):
        f.append("judge_rejected")
    if not f:
        return f, "verified"
    return f, "rejected" if any(x in ic["reject_on"] for x in f) else "uncertain"


def interpret_pair(d, cfg, client):
    vc, ic, pr = cfg["vlm"], cfg["interpret"], cfg["prompts"]
    d = Path(d)
    meta = load_json(d / "pair.json")
    regs = load_json(d / "regions.json")["regions"]
    with rasterio.open(d / "before.tif") as f:
        a, tf, crs = f.read(), f.transform, f.crs
    with rasterio.open(d / "after.tif") as f:
        b = f.read()
    with rasterio.open(d / "prob.tif") as f:
        prob = f.read(1)
    with rasterio.open(d / "labels.tif") as f:
        lab = f.read(1)
    ia, ib = to_display(a, meta["modality"], vc), to_display(b, meta["modality"], vc)
    regs.sort(key=lambda r: r["area_m2"] * r["mean_prob"], reverse=True)
    fmt = dict(modality=meta["modality"], gsd=meta["res_m"], types=ic["change_types"], nuisance=ic["nuisance_types"])
    feats = []
    for n, r in enumerate(regs[: ic["max_regions"]]):
        rng = random.Random(ic["seed"] * 100003 + r["id"])
        win = window(r["bbox_px"], lab.shape, vc)
        cb, ca = chip(ia, win, vc), chip(ib, win, vc)
        ov = chip(ib, win, vc, r["polygon_px"])
        ovb = chip(ia, win, vc, r["polygon_px"])
        bd, ad = meta["before_date"][:10], meta["after_date"][:10]
        ask = lambda x, y, dx, dy, xo, p=pr["describe"]: parse_json(call_vlm(
            client, vc["model"], pr["system"],
            build_content(p.format(before_date=dx, after_date=dy, **fmt),
                          [("BEFORE", x), ("AFTER", y), ("AFTER with region outlined", xo)]), vc["max_tokens"]))
        desc = ask(cb, ca, bd, ad, ov)
        swap = ask(ca, cb, ad, bd, ovb) if rng.random() < ic["swap_rate"] else None
        null = ask(cb, cb, bd, bd, ovb) if rng.random() < ic["null_rate"] else None
        judge = None
        if ic["judge"] and desc:
            claim = json.dumps({k: desc.get(k) for k in ("change_type", "object", "before_state", "after_state", "description")})
            judge = parse_json(call_vlm(
                client, vc["judge_model"], pr["system"],
                build_content(pr["judge"].format(before_date=bd, after_date=ad, claim=claim),
                              [("BEFORE", cb), ("AFTER", ca), ("AFTER with region outlined", ov)]), vc["max_tokens"]))
        evidence = {"mean_prob": r["mean_prob"], "contrast": contrast(prob, lab, r, ic["ring_px"])}
        fails, status = evaluate(desc, swap, null, judge, evidence, ic)
        feats.append({
            "type": "Feature",
            "geometry": r["geometry_lonlat"],
            "properties": {"region_id": r["id"], "area_m2": r["area_m2"], **evidence, "status": status, "failures": fails,
                           "description": desc, "swap_check": swap, "null_check": null, "judge": judge},
        })
        log.info("%s region %s -> %s %s", meta["pair_id"], r["id"], status, fails)
    summary = None
    if ic["summarize"]:
        keep = [{"type": x["properties"]["description"].get("change_type"), "text": x["properties"]["description"].get("description"),
                 "status": x["properties"]["status"]} for x in feats if x["properties"]["status"] != "rejected" and x["properties"]["description"]]
        if keep:
            summary = call_vlm(client, vc["model"], pr["system"], [{"type": "text", "text": pr["summary"].format(
                before_date=meta["before_date"][:10], after_date=meta["after_date"][:10], findings=json.dumps(keep))}], vc["max_tokens"])
    rep = Path(cfg["paths"]["reports"]) / meta["pair_id"]
    rep.mkdir(parents=True, exist_ok=True)
    counts = {s: sum(1 for x in feats if x["properties"]["status"] == s) for s in ("verified", "uncertain", "rejected")}
    dump_json({"type": "FeatureCollection", "features": feats,
               "pair": {k: meta[k] for k in ("pair_id", "source", "modality", "before_id", "after_id", "before_date", "after_date", "gap_days", "license")},
               "counts": counts, "summary": summary}, rep / "report.geojson")
    return counts


def run(cfg, pair_dirs):
    client = Backend(cfg["vlm"])
    return {str(d): interpret_pair(d, cfg, client) for d in pair_dirs}

### `build_dataset.py`

In [ ]:
%%writefile build_dataset.py
import argparse
import copy
import csv
from pathlib import Path
import pandas as pd
import rasterio
from common import ensure, get_logger, load_config, load_json
import discover
import pairing
import public_sets
from interpret import to_display
from pair_io import write_pair

log = get_logger("build_dataset")

ADAPTERS = {
    "levir_cd": [("Hugging Face mirror", public_sets.levir_hf), ("local folders", public_sets.paired_folders)],
    "whu_cd": [("local folders", public_sets.paired_folders)],
    "oscd": [("local OSCD folder", public_sets.oscd_local)],
    "xbd": [("local xBD folder", public_sets.xbd_local)],
    "spacenet7": [("public AWS bucket", public_sets.spacenet7_s3)],
}


def site_config(cfg, dcfg, site, k, windows):
    c = copy.deepcopy(cfg)
    base = Path(dcfg["sentinel2"]["work_dir"]) / site["name"] / f"try{k}"
    c["aoi"]["bbox"] = site["bbox"]
    c["pairing"]["windows"] = windows
    c["pairing"]["match_histograms"] = dcfg["sentinel2"]["match_histograms"]
    c["pairing"]["max_pairs_per_source"] = dcfg["sentinel2"]["pairs_per_site"]
    c["paths"] = {"work": str(base), "catalog": str(base / "catalog.csv"), "pairs": str(base / "pairs.csv"),
                  "prepared": str(base / "prepared"), "reports": str(base / "reports")}
    return c


def write_site_pairs(c, site, dcfg, oc):
    sc = dcfg["sentinel2"]
    written = 0
    for d in pairing.accepted_dirs(c)[: sc["pairs_per_site"]]:
        meta = load_json(d / "pair.json")
        with rasterio.open(d / "before.tif") as f:
            a = f.read()
        with rasterio.open(d / "after.tif") as f:
            b = f.read()
        ia, ib = to_display(a, meta["modality"], c["vlm"]), to_display(b, meta["modality"], c["vlm"])
        bd, ad = meta["before_date"][:10], meta["after_date"][:10]
        m = {"dataset": "Sentinel-2 L2A", "site": site["name"], "note": site["note"], "before_date": bd, "after_date": ad,
             "before_scene": meta["before_id"], "after_scene": meta["after_id"], "gsd_m": meta["res_m"],
             "crs": meta["crs"], "bbox_lonlat": site["bbox"], "source_url": sc["source_url"], "license": sc["license"],
             "processing": "windowed read of cloud-optimised GeoTIFFs, reprojected to UTM, per-band percentile stretch for display; no detections or outlines drawn"}
        write_pair(oc["root"], f"sentinel2/{site['name']}/{bd}_{ad}", ia, ib, m, oc)
        written += 1
    return written


def run_site(cfg, dcfg, site, oc):
    notes, written = [], 0
    for k, windows in enumerate(site["tries"]):
        label = f"try {k + 1} (before {windows['before']}, after {windows['after']})"
        c = site_config(cfg, dcfg, site, k, windows)
        try:
            cat = discover.run(c)
            if len(cat) == 0:
                notes.append(f"{label}: the scene search returned 0 scenes")
                continue
            pairing.run(c)
            written = write_site_pairs(c, site, dcfg, oc)
        except SystemExit as e:
            notes.append(f"{label}: {e}")
            continue
        except Exception as e:
            notes.append(f"{label}: {type(e).__name__}: {e}")
            continue
        if written:
            break
        df = pd.read_csv(c["paths"]["pairs"])
        notes.append(f"{label}: no pair passed verification; reasons: {df['reason'].value_counts().to_dict() if 'reason' in df else 'none recorded'}")
    return {"status": "ok" if written else "failed", "pairs": written, "notes": notes}


def run_public(key, c, oc):
    pct = oc["display_pct"]
    notes, written = [], 0
    for label, fn in ADAPTERS[key]:
        try:
            for item in fn(c, pct):
                meta = {"dataset": c["name"], "kind": c["kind"], "source_url": c["source_url"], "license": c["license"],
                        "date_note": c["date_note"], "before_date": c.get("before_date"), "after_date": c.get("after_date")}
                meta.update({k: v for k, v in item["meta"].items() if v is not None})
                write_pair(oc["root"], f"{key}/{item['rel']}", item["before"], item["after"], meta, oc)
                written += 1
        except Exception as e:
            notes.append(f"{label}: {type(e).__name__}: {e}")
        if written:
            break
    status = "ok" if written else "failed"
    if status == "failed":
        notes.append(f"How to obtain: {c['how_to_obtain']}")
    return {"status": status, "pairs": written, "notes": notes}


def write_report(results, dcfg):
    oc = dcfg["output"]
    lines = ["# Download report", "", "Generated by `build_dataset.py`. Every failure below is the exact message raised on the machine that ran the build.", "",
             "| Source | Status | Pairs saved |", "|---|---|---|"]
    lines += [f"| {k} | {v['status']} | {v['pairs']} |" for k, v in results.items()]
    for k, v in results.items():
        if v["notes"]:
            lines += ["", f"## {k}", ""] + [f"- {n}" for n in v["notes"]]
    Path(oc["report_file"]).write_text("\n".join(lines) + "\n", encoding="utf-8")


def write_index(oc):
    rows = []
    for mj in sorted(Path(oc["root"]).rglob("meta.json")):
        m = load_json(mj)
        rows.append([mj.parent.relative_to(oc["root"]).as_posix(), m.get("dataset"), m.get("before_date"), m.get("after_date")])
    with open(oc["index_file"], "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["folder", "dataset", "before_date", "after_date"])
        w.writerows(rows)
    return len(rows)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--dataset-config", required=True)
    ap.add_argument("--only", nargs="*")
    args = ap.parse_args()
    cfg, dcfg = load_config(args.config), load_config(args.dataset_config)
    oc = dcfg["output"]
    ensure(oc["root"])
    results = {}
    if dcfg["sentinel2"]["enabled"] and (not args.only or "sentinel2" in args.only):
        for site in dcfg["sentinel2"]["sites"]:
            results[f"sentinel2/{site['name']}"] = run_site(cfg, dcfg, site, oc)
    for key, c in dcfg["public"].items():
        if c["enabled"] and (not args.only or key in args.only):
            results[key] = run_public(key, c, oc)
    write_report(results, dcfg)
    log.info("pairs in index: %d", write_index(oc))
    for k, v in results.items():
        log.info("%s: %s, %d pairs", k, v["status"], v["pairs"])


if __name__ == "__main__":
    main()

## Run

In [ ]:
!python build_dataset.py --config config.yaml --dataset-config dataset.yaml

## Review

In [ ]:
import pandas as pd
print(open('dataset/DOWNLOAD_REPORT.md').read())
pd.read_csv('dataset/INDEX.csv')